## Librerías

In [1]:
import pandas as pd
import joblib
import time
import numpy as np
from sklearn.neighbors import NearestNeighbors

### Carga de objetos

In [2]:
pathModelos = "/content/drive/MyDrive/AnteproyectoTerminal/Código/Modelos/"

In [17]:
scaler = joblib.load(pathModelos+'standardScaler_21abril2026.joblib')
kmeans = joblib.load(pathModelos+'modeloKmeans_21abril2026.joblib')
dict_modelos_knn = joblib.load(pathModelos+'modelosKnnClusters_21abril2026.joblib')
pca = joblib.load(pathModelos+'modeloPCA_21abril2026.joblib')

## Carga de objetos

In [35]:
# Df de entrenamiento (Contiene las 28 dims + cluster_label)
df_train_pca = pd.read_csv(pathModelos+'df_train_pca.csv', index_col=0)
# Consultas de prueba (Contiene las 55 dims escaladas)
df_test_scaled = pd.read_csv(pathModelos+'df_test_scaled.csv', index_col=0)

In [36]:
print("✅ Todos los objetos cargados exitosamente.")
print(f"Catálogo: {df_train_pca.shape[0]} canciones | Consultas: {df_test_scaled.shape[0]} canciones.")

✅ Todos los objetos cargados exitosamente.
Catálogo: 18766 canciones | Consultas: 4692 canciones.


In [37]:
X_test_scaled = df_test_scaled.values

In [42]:
X_train_scaled_pca = df_train_pca.drop('cluster_label', axis=1).values

## Simulación

In [38]:
def recomendar_canciones_hibridas(song, modelo_kmeans, dict_modelos_knn, df_train, k=3):
    song_reshaped = song.reshape(1, -1)

    # Identificar el clúster
    cluster_asignado = modelo_kmeans.predict(song_reshaped)[0]

    # Modelo KNN pre-entrenado para ese clúster específico
    knn_modelo = dict_modelos_knn[cluster_asignado]

    # Búsqueda K-NN Local k+1 para poder filtrar la canción de origen si es necesario
    distancias, indices = knn_modelo.kneighbors(song_reshaped, n_neighbors=k+1)

    # IDs reales desde el subset del DataFrame original
    df_cluster_subset = df_train[df_train['cluster_label'] == cluster_asignado]

    # Extracción de índices basándonos en la posición local devuelta por el KNN
    ids_recomendados = df_cluster_subset.index[indices[0]].tolist()

    # k canciones más cercanas (quitando la primera si es la misma)
    return ids_recomendados[1:k+1]




Aplicar PCA en el test

Seleccionar 2 canciones aleatorias del set de TEST

In [39]:
n_casos = 2
indices_seeds = np.random.choice(df_test_scaled.index, n_casos, replace=False)

In [40]:
for i, seed_idx in enumerate(indices_seeds):
    print(f"CASO {i+1}:")
    print(f"Canción Semilla (ID): {seed_idx}")

    x_raw = df_test_scaled.loc[[seed_idx]].values

    # Transformar de 56 a 28 dimensiones
    x_query_pca = pca.transform(x_raw)

    try:
        # Recomendación de tu Modelo
        recs_ids = recomendar_canciones_hibridas(x_query_pca[0], kmeans, dict_modelos_knn, df_train_pca, k=3)
        print(f"-> Recomendaciones del Modelo: {recs_ids}")

        # Recomendación Aleatoria
        rec_aleatoria = df_train_pca.sample(1).index.tolist()
        print(f"-> Recomendación Aleatoria: {rec_aleatoria}")

    except ValueError as e:
        print(f"❌ Error en las dimensiones: {e}")

    print("-" * 40)

CASO 1:
Canción Semilla (ID): 131652
-> Recomendaciones del Modelo: [84289, 19644, 131552]
-> Recomendación Aleatoria: [122836]
----------------------------------------
CASO 2:
Canción Semilla (ID): 133429
-> Recomendaciones del Modelo: [59047, 68743, 11696]
-> Recomendación Aleatoria: [110863]
----------------------------------------


## Evaluación

### Evaluación de la Calidad de la Recomendación (Distancia y Eficiencia)

In [45]:
df_train_pca

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC20,PC21,PC22,PC23,PC24,PC25,PC26,PC27,PC28,cluster_label
id,,,,,,,,,,,,,,,,,,,,,
118626,0.227098,1.387593,0.000770,-0.180226,-2.702992,-0.482546,0.336372,-0.023594,0.922912,-1.535967,...,-0.463216,-0.246745,-0.717319,0.431190,-0.060068,-0.108449,0.159385,-0.626649,0.022212,0
138013,0.337013,0.740272,-1.479094,-1.057334,0.675970,0.426453,-0.000479,-0.475699,-1.302179,1.958907,...,0.058582,-0.485072,-0.049986,0.165740,-0.258217,0.672340,0.405699,-0.816574,-0.963568,0
55289,1.914273,0.131263,2.838959,-0.508145,0.824533,-0.743111,-3.151662,-0.952272,-0.309910,0.631152,...,0.760118,-0.025054,0.692607,-0.596188,0.146698,0.438598,-0.830631,-0.217872,-0.420926,0
44193,-0.921334,-3.122082,-2.531104,-0.763391,-1.048050,1.712361,0.560527,0.225879,0.270235,1.844875,...,0.466549,0.766709,-0.336390,0.054829,0.081287,0.177832,0.914408,0.449963,0.503000,1
1555,-2.402274,-4.610536,-5.177472,-0.669809,0.943811,0.272341,0.288731,0.513308,0.150114,-1.080870,...,-1.977537,-0.240679,-0.320931,0.458482,-0.066328,0.622134,0.105078,-0.007194,-0.257786,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76554,-4.767483,2.578849,1.641820,-0.066753,-0.530852,-0.248828,0.962056,0.474092,-0.040335,-0.389067,...,-0.314530,-0.630899,-0.338395,0.623762,0.046102,0.354177,-0.196100,-0.022418,-0.946567,2
139990,-2.602044,-2.512520,-1.701372,-1.837892,-1.727555,-0.562507,1.163866,0.625077,0.065284,0.181662,...,0.062895,0.284418,-0.216523,-0.160399,-0.002699,-0.805329,-0.592363,0.437728,0.169896,1
37046,5.074806,-3.074942,1.595560,-1.511884,0.664206,1.933072,-0.862222,-1.155801,-0.036753,0.800042,...,1.009363,-0.505261,0.574771,-0.476956,0.789305,0.351433,0.018895,-0.328620,-0.301289,4


## Evaluación para el proceso de anexo de nueva canción

In [46]:
# Función para recomendación aleatoria (Baseline)
def recomendar_aleatorio(df_train, k=3):
    return df_train.sample(k).index.tolist()

# Configuración de la prueba (100 consultas para promediar)
n_pruebas = 100
indices_test = np.random.choice(len(X_test_scaled_pca), n_pruebas, replace=False)

latencias_hibrido = []
latencias_estandar = []
distancias_hibrido = []
distancias_random = []

# Preparar KNN Estándar para la comparativa
knn_estandar = NearestNeighbors(n_neighbors=2, metric='cosine', algorithm='brute').fit(X_train_scaled_pca)

for idx in indices_test:
    # 1. Obtenemos la canción semilla (56 dims)
    x_raw = X_test_scaled_pca[idx].reshape(1, -1)

    x_query = pca.transform(x_raw)

    # --- Prueba Híbrido ---
    start = time.time()
    recs_hibrido_ids = recomendar_canciones_hibridas(x_query[0], kmeans, dict_modelos_knn, df_train_pca, k=3)
    latencias_hibrido.append(time.time() - start)

    # --- Medir distancias ---
    recs_vectores = df_train_pca.loc[recs_hibrido_ids].drop('cluster_label', axis=1).values
    dist_h = np.mean([np.linalg.norm(x_query - r) for r in recs_vectores])
    distancias_hibrido.append(dist_h)

    # --- Prueba K-NN Estándar ---
    start = time.time()
    _, _ = knn_estandar.kneighbors(x_query)
    latencias_estandar.append(time.time() - start)

    # Prueba Aleatoria (Baseline de precisión)
    recs_rand_ids = recomendar_aleatorio(df_train_pca, k=3)
    recs_rand_vectores = df_train_pca.loc[recs_rand_ids].drop('cluster_label', axis=1).values
    dist_r = np.mean([np.linalg.norm(x_query - r) for r in recs_rand_vectores])
    distancias_random.append(dist_r)

print(f"--- Comparativa de Eficiencia (Latencia) ---")
print(f"Tiempo promedio Híbrido: {np.mean(latencias_hibrido):.6f} seg")
print(f"Tiempo promedio K-NN Estándar: {np.mean(latencias_estandar):.6f} seg")
print(f"Mejora de velocidad: {np.mean(latencias_estandar) / np.mean(latencias_hibrido):.2f}x más rápido")

print(f"\n--- Comparativa de Precisión (cosine) ---")
print(f"Distancia promedio Híbrido: {np.mean(distancias_hibrido):.4f}")
print(f"Distancia promedio Aleatorio: {np.mean(distancias_random):.4f}")
print(f"Mejora en cercanía acústica: {((np.mean(distancias_random) - np.mean(distancias_hibrido)) / np.mean(distancias_random)) * 100:.2f}%")

--- Comparativa de Eficiencia (Latencia) ---
Tiempo promedio Híbrido: 0.004261 seg
Tiempo promedio K-NN Estándar: 0.005243 seg
Mejora de velocidad: 1.23x más rápido

--- Comparativa de Precisión (cosine) ---
Distancia promedio Híbrido: 4.2428
Distancia promedio Aleatorio: 9.5998
Mejora en cercanía acústica: 55.80%


## Guardar para Flask

In [ ]:
modeloKmeans = 'modeloKmeans_20abril2026.joblib'
standardScaler = 'standardScaler_20abril2026.joblib'
modelosKnn = 'modelosKnnClusters_20abril2026.joblib'
# Guardar el modelo entrenado
joblib.dump(kmeans_final, modeloKmeans)

# Guardar el escalador (ej. RobustScaler o StandardScaler)
joblib.dump(standard_scaler, standardScaler)

# Modelos knn para cada cluster
joblib.dump(dict_modelos_knn, modelosKnn)

['modelosKnnClusters_20abril2026.joblib']

In [ ]:
print(f" Diccionario KNN generado con {len(features_audio)} características.")

 Diccionario KNN generado con 28 características.


## Extractor de enlaces de drive

In [ ]:
# Montar Drive
drive.mount('/content/drive')

# Autenticación para usar la API de Drive
auth.authenticate_user()
drive_service = build('drive', 'v3')

In [ ]:
# https://drive.google.com/drive/folders/1JW4XDk26TeHYUGoYzZoXAb_JF-EKozaC?usp=drive_link
FOLDER_ID = '1JW4XDk26TeHYUGoYzZoXAb_JF-EKozaC'

def get_drive_files(folder_id):
    query = f"'{folder_id}' in parents and trashed = false"
    results = drive_service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    return results.get('files', [])

In [ ]:
all_music_data = []

subfolders = get_drive_files(FOLDER_ID)

In [ ]:
contador = 0

In [ ]:
def get_all_subfolders(parent_id):
    query = f"'{parent_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    subfolders = []
    page_token = None

    while True:

        response = drive_service.files().list(
            q=query,
            spaces='drive',
            fields='nextPageToken, files(id, name, mimeType)', # <-- AGREGAR mimeType AQUÍ
            pageToken=page_token
        ).execute()

        subfolders.extend(response.get('files', []))
        page_token = response.get('nextPageToken')

        if not page_token:
            break

    return subfolders

subfolders = get_all_subfolders(FOLDER_ID)
print(len(subfolders))

In [ ]:
for sub in subfolders:
    if sub['mimeType'] == 'application/vnd.google-apps.folder':
        print(f"Procesando carpeta: {sub['name']}")
        files_in_sub = get_drive_files(sub['id'])

        for f in files_in_sub:
            if f['name'].endswith('.mp3'):
              contador = contador+1
                #track_id = f['name'].split('.')[0]

                # Crear el enlace de reproducción directa
                #direct_link = f"https://drive.google.com/uc?id={f['id']}&export=download"

                #all_music_data.append({
                 #   'track_id': track_id,
                  #  'drive_id': f['id'],
                  #  'stream_url': direct_link,
                  #  'folder': sub['name']
               # })

In [ ]:
contador

In [ ]:
mapeoFile = 'mapeo_drive.csv'
mapping_df = pd.DataFrame(all_music_data)
mapping_df.to_csv(mapeoFile, index=False)
print("Rutas listas")

## Descarga de archivos

In [ ]:
# Descargar modelo
descargar = True
if descargar:
  files.download(modeloKmeans)
  files.download(standardScaler)
  files.download(modelosKnn)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>